# Inventory Analysis Capstone

Start by importing all necessary Python libraries

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import requests
import ast
import psycopg2
import sqlalchemy


### Data Acquisition - Inventory


Read in the dataset with the 500 companies first

In [2]:
companies = pd.read_csv('data/SP500.csv')
companies.head(5)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373


Collect Net Inventory from SEC filings for each company

In [3]:
def fetch_sec_data(cik):
    headers = {'User-Agent' : 'marshalln@etown.edu', 'Host': 'data.sec.gov'}
    cik = str(cik).zfill(10)
    url = 'https://data.sec.gov/api/xbrl/companyfacts/CIK' + cik + '.json'
    response = requests.get(url,headers=headers)
    response.raise_for_status()
    try:
        inventory = response.json()['facts']['us-gaap']['InventoryNet']['units']['USD']
    except:
        inventory = None
    try:
        cogs = response.json()['facts']['us-gaap']['CostOfGoodsSold']['units']['USD']
    except: 
        try:
            cogs = response.json()['facts']['us-gaap']['CostOfGoodsAndServicesSold']['units']['USD']
        except:
            try:
                cogs = response.json()['facts']['us-gaap']['CostOfRevenue']['units']['USD']
            except:
                try:
                    cogs = response.json()['facts']['us-gaap']['CostOfSales']['units']['USD']
                except:
                    cogs = None

    return (inventory, cogs)


In [6]:
data = pd.DataFrame(columns=['company', 'industry', 'inventory', 'cogs'])
for i in range(len(companies)):
    symbol = companies['Symbol'][i]
    sec_data = fetch_sec_data(companies['CIK'][i])
    if sec_data[0] and sec_data[1] is not None:
        data.loc[len(data)] = [symbol, companies['GICS Sub-Industry'][i], sec_data[0], sec_data[1]]
data.head()

,company,industry,inventory,cogs
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'val': 3013000000, 'acc...","[{'start': '2007-01-01', 'end': '2007-12-31', ..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'val': 215100000, 'accn...","[{'start': '2008-01-01', 'end': '2008-12-31', ..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'val': 2951442000, 'acc...","[{'start': '2007-01-01', 'end': '2007-12-31', ..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'val': 1091000000, 'acc...","[{'start': '2011-01-01', 'end': '2011-12-31', ..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'val': 567000000, 'accn...","[{'start': '2007-12-30', 'end': '2008-12-27', ..."


Export to a csv file for easy use later

In [7]:
data.to_csv('data/financial_data.csv', index=False)

### Data Exploration

In [2]:
financial_data = pd.read_csv('data/financial_data.csv')

Observe the shape of the data

In [9]:
print(financial_data.shape)

(281, 4)


Describe the data

In [10]:
print(financial_data.describe())

       company               industry  \
count      281                    281   
unique     281                     87   
top        MMM  Health Care Equipment   
freq         1                     16   

                                                inventory  \
count                                                 281   
unique                                                280   
top     [{'end': '2015-12-31', 'val': 491000000, 'accn...   
freq                                                    2   

                                                     cogs  
count                                                 281  
unique                                                280  
top     [{'start': '2013-01-01', 'end': '2013-12-31', ...  
freq                                                    2  


Example of the data

In [11]:
financial_data.head(5)

,company,industry,inventory,cogs
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'val': 3013000000, 'acc...","[{'start': '2007-01-01', 'end': '2007-12-31', ..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'val': 215100000, 'accn...","[{'start': '2008-01-01', 'end': '2008-12-31', ..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'val': 2951442000, 'acc...","[{'start': '2007-01-01', 'end': '2007-12-31', ..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'val': 1091000000, 'acc...","[{'start': '2011-01-01', 'end': '2011-12-31', ..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'val': 567000000, 'accn...","[{'start': '2007-12-30', 'end': '2008-12-27', ..."


Check for nulls

In [12]:
financial_data.isnull().sum()

company      0
industry     0
inventory    0
cogs         0
dtype: int64

Check for duplicates

In [13]:
financial_data.duplicated().value_counts()

False    281
Name: count, dtype: int64

### Data Cleaning

Remove duplicate & unnecessary information from inventory and cogs columns

In [3]:
def clean_inventory(data, metric): 
    data_list = ast.literal_eval(data)
    date = ''
    new_data = [] 
    for dict in data_list:
        if dict['end'] != date: 
            date = dict['end'] 
            temp = {'end' : date, metric : dict['val']}
            new_data.append(temp) 
    return new_data

In [4]:
financial_data['inventory'] = financial_data['inventory'].apply(clean_inventory, args=('inventory',))
financial_data['cogs'] = financial_data['cogs'].apply(clean_inventory, args=('cogs',))
financial_data.head(5)

,company,industry,inventory,cogs
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'inventory': 3013000000...","[{'end': '2007-12-31', 'cogs': 12735000000}, {..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'inventory': 215100000}...","[{'end': '2008-12-31', 'cogs': 1077200000}, {'..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'inventory': 2951442000...","[{'end': '2007-12-31', 'cogs': 11422046000}, {..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'inventory': 1091000000...","[{'end': '2011-12-31', 'cogs': 4639000000}, {'..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'inventory': 567000000}...","[{'end': '2008-12-27', 'cogs': 3488000000}, {'..."


Create separate date and inventory/cogs columns

In [12]:
def split_data(row):
    df = pd.DataFrame(columns=['company','industry', 'inventory', 'cogs', 'date'])
    for inventory, cogs in zip(row['inventory'], row['cogs']):
        df.loc[len(df)] = [row['company'], row['industry'], inventory['inventory'], cogs['cogs'], pd.to_datetime(inventory['end'])]
    return df

In [17]:
clean_financial_data = pd.DataFrame(columns = ['company', 'industry', 'inventory', 'cogs', 'date'])
for i in range(len(financial_data)):
    clean_financial_data = pd.concat([clean_financial_data, split_data(financial_data.iloc[i])], ignore_index=True)

Briefly inspect the cleaned data

In [18]:
print(len(clean_financial_data))

10598


In [19]:
clean_financial_data.tail()

,company,industry,inventory,cogs,date
10593,ZTS,Pharmaceuticals,2306000000,2561000000,2024-12-31 00:00:00
10594,ZTS,Pharmaceuticals,2365000000,643000000,2025-03-31 00:00:00
10595,ZTS,Pharmaceuticals,2439000000,1311000000,2025-06-30 00:00:00
10596,ZTS,Pharmaceuticals,2465000000,2012000000,2025-09-30 00:00:00
10597,ZTS,Pharmaceuticals,2430000000,2719000000,2025-12-31 00:00:00


### Data Aquisition pt 2

Use the current data to grab stock data

In [14]:
def acquire_stock_data(row):
    ticker = row['company']
    for i in range(5): # in case date is on weekend of holiday, loop back to first available date
        date = pd.to_datetime(row['date'])
        interval = pd.Timedelta(days=i)
        try:
            stock = yf.download(tickers=ticker, start=date-interval, end=date-interval+pd.Timedelta(days=1))['Close'][ticker][date]#[str(row['date'].date())]
            return stock
        except:
            pass
    return None 


In [15]:
print(acquire_stock_data(clean_inventory.iloc[0]))

[*********************100%***********************]  1 of 1 completed

28.680505752563477


In [ ]:
stocks_data = pd.DataFrame(columns=['company', 'date', 'price'])

for i in range(10):
    stocks_data.loc[len(stocks_data)] = [clean_inventory.iloc['row'], clean_inventory.iloc['date'], acquire_stock_data(clean_inventory.iloc[i])]

stocks_data.head(5)

NameError: name 'pd' is not defined